# Grid <-> particle-set round-trip

Seed a grid, emit it as a flat particle set, then ingest advected positions
back onto the grid -- the package's Parcels boundary, exercised here *without*
Parcels. We feed the emitted positions straight back, so the round trip is the
identity and we can check `to_parcels_pset` / `from_parcels_pset_lon_lat` are
lossless inverses.

In [1]:
import numpy as np

from lcs_parcels import AuxiliaryGrid, NeighborGrid

t0 = np.datetime64("2020-01-01")
t1 = np.datetime64("2020-01-08")  # +7 day window; sign(T) > 0 -> repelling LCS
lon_axis = np.linspace(-25.0, -20.0, 6)
lat_axis = np.linspace(15.0, 20.0, 5)

## Seed a NeighborGrid

`from_axes` broadcasts the 1-D axes into curvilinear reference positions
`lon_0(i, j)` / `lat_0(i, j)` and records the release time `t0`.

In [2]:
seed = NeighborGrid.from_axes(lon_axis, lat_axis, t0=t0)
seed.ds

<xarray.Dataset> Size: 1kB
Dimensions:  (i: 6, j: 5)
Coordinates:
  * i        (i) int64 48B 0 1 2 3 4 5
  * j        (j) int64 40B 0 1 2 3 4
    lon_0    (i, j) float64 240B -25.0 -25.0 -25.0 -25.0 ... -20.0 -20.0 -20.0
    lat_0    (i, j) float64 240B 15.0 16.25 17.5 18.75 ... 16.25 17.5 18.75 20.0
    t0       datetime64[s] 8B 2020-01-01
    T        timedelta64[s] 8B 00:00:00
Data variables:
    lon      (i, j) float64 240B -25.0 -25.0 -25.0 -25.0 ... -20.0 -20.0 -20.0
    lat      (i, j) float64 240B 15.0 16.25 17.5 18.75 ... 16.25 17.5 18.75 20.0

## Emit a particle set

`to_parcels_pset` flattens the reference positions to plain `(lon, lat)` lists,
one entry per grid point -- ready to hand to a Parcels `ParticleSet`.

In [3]:
lon, lat = seed.to_parcels_pset()
print(f"{len(lon)} particles ({lon_axis.size} x {lat_axis.size})")
print("first three lon:", lon[:3])

30 particles (6 x 5)
first three lon: [np.float64(-25.0), np.float64(-25.0), np.float64(-25.0)]


## Ingest advected positions

`from_parcels_pset_lon_lat` reattaches the flat positions and derives the
signed window `T = t1 - t0`. With Parcels these would be the advected outputs;
here they are the emitted positions, so the ingested grid equals the seed.

In [4]:
advected = NeighborGrid.from_parcels_pset_lon_lat(seed, lon, lat, t1=t1)
advected.ds

<xarray.Dataset> Size: 1kB
Dimensions:  (i: 6, j: 5)
Coordinates:
  * i        (i) int64 48B 0 1 2 3 4 5
  * j        (j) int64 40B 0 1 2 3 4
    lon_0    (i, j) float64 240B -25.0 -25.0 -25.0 -25.0 ... -20.0 -20.0 -20.0
    lat_0    (i, j) float64 240B 15.0 16.25 17.5 18.75 ... 16.25 17.5 18.75 20.0
    t0       datetime64[s] 8B 2020-01-01
    T        timedelta64[s] 8B 7 days
Data variables:
    lon      (i, j) float64 240B -25.0 -25.0 -25.0 -25.0 ... -20.0 -20.0 -20.0
    lat      (i, j) float64 240B 15.0 16.25 17.5 18.75 ... 16.25 17.5 18.75 20.0

Round-trip checks: the `(i, j)` grid is recovered, `T` is the signed window,
and the advected `lon`/`lat` match the reference positions (identity input).

In [5]:
print("dims:", dict(advected.ds.sizes))
print("T:", advected.ds["T"].values)
print("lon matches reference:", np.allclose(advected.ds["lon"], seed.ds["lon_0"]))
print("lat matches reference:", np.allclose(advected.ds["lat"], seed.ds["lat_0"]))

dims: {'i': 6, 'j': 5}
T: 604800 seconds
lon matches reference: True
lat matches reference: True


## AuxiliaryGrid: four arms per point

The auxiliary stencil places four arms (`displacement = [east, north, west,
south]`) around each grid point. The arms are the *explicit* reference
positions `lon_0`/`lat_0` -- so the dataset is self-sufficient -- and the
grid-point centres (where the FTLE is reported) are kept separately as
`lon_c`/`lat_c`.

In [6]:
aux_seed = AuxiliaryGrid.from_axes(lon_axis, lat_axis, t0=t0)
aux_lon, aux_lat = aux_seed.to_parcels_pset()
print(f"{len(aux_lon)} particles = {lon_axis.size} x {lat_axis.size} x 4 arms")

120 particles = 6 x 5 x 4 arms


In [7]:
aux = AuxiliaryGrid.from_parcels_pset_lon_lat(aux_seed, aux_lon, aux_lat, t1=t1)
print("reference arms  lon_0:", aux.ds["lon_0"].dims)
print("advected arms   lon  :", aux.ds["lon"].dims)
print("diagnostic centres lon_c:", aux.ds["lon_c"].dims)

reference arms  lon_0: ('i', 'j', 'displacement')
advected arms   lon  : ('i', 'j', 'displacement')
diagnostic centres lon_c: ('i', 'j')


Re-emitting the ingested auxiliary grid reproduces the same particle set --
`to_parcels_pset` emits the explicit `lon_0`/`lat_0` arms directly.

In [8]:
re_lon, re_lat = aux.to_parcels_pset()
print("re-emitted pset matches:", np.allclose(re_lon, aux_lon) and np.allclose(re_lat, aux_lat))

re-emitted pset matches: True
